In [1]:
import os
for folder in ["ingestion", "retrieval", "generation", "ui"]:
    os.makedirs(folder, exist_ok=True)
    open(f"{folder}/__init__.py", "a").close()
print("Created:", os.listdir())

Created: ['.config', 'ui', 'generation', 'retrieval', 'ingestion', 'sample_data']


In [2]:
%%writefile config.py
MODEL_NAME = "gemini-2.0-flash"
CHUNK_SIZE = 300       # characters per chunk
CHUNK_OVERLAP = 50     # overlap between chunks
TOP_K = 3              # how many chunks to retrieve per question
COLLECTION_NAME = "capstone_docs"

Writing config.py


In [3]:
%%writefile README.md
# Capstone: Document Q&A Assistant (RAG)

## What it does
Answers questions about a document set using Retrieval-Augmented Generation.

## Document set
(fill in once you pick your real docs)

## Architecture
text -> chunks -> embeddings -> vector DB -> retrieve top-k -> LLM answer

## Components checklist
- [ ] Chunking (Day 18)
- [ ] Embed + store (Day 19)
- [ ] Retrieval + generation (Day 20)
- [ ] Citations + refusal (Day 21)

Writing README.md


In [4]:
%%writefile ingestion/chunker.py
from config import CHUNK_SIZE, CHUNK_OVERLAP

def load_text_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap   # overlap = don't lose ideas at boundaries
    return chunks

Writing ingestion/chunker.py


In [5]:
%%writefile sample_doc.txt
Refunds are processed within 5-7 business days after a cancellation request. You can cancel your plan anytime from the account settings page. Our support team is available 24/7 via live chat and email. To reset your password, click 'Forgot Password' on the login screen. We offer a 14-day free trial with full access to all premium features. Enterprise plans include dedicated account managers and custom SLAs.

Writing sample_doc.txt


In [6]:
from ingestion.chunker import load_text_file, chunk_text

text = load_text_file("sample_doc.txt")
chunks = chunk_text(text)
print(f"Created {len(chunks)} chunks\n")
for c in chunks:
    print("---")
    print(c)

Created 2 chunks

---
Refunds are processed within 5-7 business days after a cancellation request. You can cancel your plan anytime from the account settings page. Our support team is available 24/7 via live chat and email. To reset your password, click 'Forgot Password' on the login screen. We offer a 14-day free trial
---
on the login screen. We offer a 14-day free trial with full access to all premium features. Enterprise plans include dedicated account managers and custom SLAs.


In [7]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [8]:
%%writefile ingestion/embed_store.py
import chromadb
from config import COLLECTION_NAME

client = chromadb.PersistentClient(path="./chroma_db")  # saves to disk
collection = client.get_or_create_collection(name=COLLECTION_NAME)

def store_chunks(chunks, source_name):
    # Deterministic IDs + upsert = re-running this never creates duplicates
    ids = [f"{source_name}_chunk_{i:04d}" for i in range(len(chunks))]
    metadatas = [{"source": source_name, "chunk_index": i} for i in range(len(chunks))]
    collection.upsert(ids=ids, documents=chunks, metadatas=metadatas)
    return len(chunks)

Writing ingestion/embed_store.py


In [9]:
from ingestion.chunker import load_text_file, chunk_text
from ingestion.embed_store import store_chunks, collection

text = load_text_file("sample_doc.txt")
chunks = chunk_text(text)
n = store_chunks(chunks, "sample_doc.txt")
print(f"Stored {n} chunks. Collection count: {collection.count()}")

# Run again to prove no duplicates appear
store_chunks(chunks, "sample_doc.txt")
print(f"After re-running: {collection.count()} (should be unchanged)")

# Test query
results = collection.query(query_texts=["How do I cancel?"], n_results=2)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(meta, "->", doc[:80])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 29.3MiB/s]


Stored 2 chunks. Collection count: 2
After re-running: 2 (should be unchanged)
{'source': 'sample_doc.txt', 'chunk_index': 0} -> Refunds are processed within 5-7 business days after a cancellation request. You
{'source': 'sample_doc.txt', 'chunk_index': 1} -> on the login screen. We offer a 14-day free trial with full access to all premiu


In [10]:
%%writefile retrieval/retriever.py
from config import TOP_K
from ingestion.embed_store import collection

def retrieve(question, k=TOP_K):
    results = collection.query(query_texts=[question], n_results=k)
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    return list(zip(docs, metas))

Writing retrieval/retriever.py


In [14]:
%%writefile generation/llm.py
import os
from config import MODEL_NAME

def call_llm(prompt, system_instruction=None):
    if not os.environ.get("GEMINI_API_KEY"):
        return f"[MOCK REPLY - no API key] Prompt was:\n{prompt[:200]}..."
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        model = genai.GenerativeModel(model_name=MODEL_NAME, system_instruction=system_instruction)
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"[ERROR calling Gemini API] {e}"

Overwriting generation/llm.py


In [15]:
%%writefile rag_answer.py
from retrieval.retriever import retrieve
from generation.llm import call_llm

SYSTEM_PROMPT = "You are a helpful assistant that answers using only the provided context."

def build_prompt(question, retrieved):
    blocks = [f"<chunk source='{m['source']}'>\n{t}\n</chunk>" for t, m in retrieved]
    context = "\n\n".join(blocks)
    return f"""Use ONLY the context below to answer.

<context>
{context}
</context>

Question: {question}
Answer:"""

def answer_question(question):
    retrieved = retrieve(question)
    prompt = build_prompt(question, retrieved)
    answer = call_llm(prompt, system_instruction=SYSTEM_PROMPT)
    return answer, retrieved

Overwriting rag_answer.py


In [16]:
import os
os.environ["GEMINI_API_KEY"] = "key"

from rag_answer import answer_question
answer, sources = answer_question("How do I cancel my plan?")
print("ANSWER:", answer)
print("\nSOURCES:")
for text, meta in sources:
    print("-", meta["source"])

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
Please retry in 50.349334946s.

In [17]:
%%writefile rag_answer.py
from retrieval.retriever import retrieve
from generation.llm import call_llm

SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say exactly: "
    "'I don't know based on the documents.' "
    "When you do answer, cite the source(s) you used by name."
)

def build_prompt(question, retrieved):
    blocks = [f"<chunk source='{m['source']}'>\n{t}\n</chunk>" for t, m in retrieved]
    context = "\n\n".join(blocks)
    return f"""Use ONLY the context below to answer. Cite the source name for any claim.
If the context doesn't contain the answer, say "I don't know based on the documents."

<context>
{context}
</context>

Question: {question}
Answer (with citation):"""

def answer_question(question):
    retrieved = retrieve(question)
    prompt = build_prompt(question, retrieved)
    answer = call_llm(prompt, system_instruction=SYSTEM_PROMPT)
    return answer, retrieved

Overwriting rag_answer.py


In [18]:
from rag_answer import answer_question

for q in ["How do I cancel my plan?", "What's the meaning of life?"]:
    answer, sources = answer_question(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

TooManyRequests: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash
Please retry in 16.274913184s.